In [25]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
import torch.nn as nn
import torchvision.models as models


In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [27]:
transform = transforms.Compose([
    transforms.Resize((150, 150)),        # Resize images to a fixed size
    transforms.ToTensor(),                # Convert image to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize
])

In [28]:
# Load the dataset
dataset_directory = "png_modified"  # Path to your image dataset
dataset = datasets.ImageFolder(root=dataset_directory, transform=transform)


In [29]:
from torch.utils.data import random_split

# Split dataset into training and validation (80/20 split)
train_size = int(0.8 * len(dataset))  # 80% for training
val_size = len(dataset) - train_size  # 20% for validation

train_data, val_data = random_split(dataset, [train_size, val_size])




In [30]:
# Print class labels to verify they're in alphabetical order
print(f"Class labels (alphabetically sorted): {dataset.classes}")


Class labels (alphabetically sorted): ['airplane', 'alarm clock', 'angel', 'ant', 'apple', 'arm', 'armchair', 'ashtray', 'axe', 'backpack', 'banana', 'barn', 'baseball bat', 'basket', 'bathtub', 'bear (animal)', 'bed', 'bee', 'beer-mug', 'bell', 'bench', 'bicycle', 'binoculars', 'blimp', 'book', 'bookshelf', 'boomerang', 'bottle opener', 'bowl', 'brain', 'bread', 'bridge', 'bulldozer', 'bus', 'bush', 'butterfly', 'cabinet', 'cactus', 'cake', 'calculator', 'camel', 'camera', 'candle', 'cannon', 'canoe', 'car (sedan)', 'carrot', 'castle', 'cat', 'cell phone', 'chair', 'chandelier', 'church', 'cigarette', 'cloud', 'comb', 'computer monitor', 'computer-mouse', 'couch', 'cow', 'crab', 'crane (machine)', 'crocodile', 'crown', 'cup', 'diamond', 'dog', 'dolphin', 'donut', 'door', 'door handle', 'dragon', 'duck', 'ear', 'elephant', 'envelope', 'eye', 'eyeglasses', 'face', 'fan', 'feather', 'fire hydrant', 'fish', 'flashlight', 'floor lamp', 'flower with stem', 'flying bird', 'flying saucer', 'f

In [31]:
# Create DataLoader objects for both training and validation sets
train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)


In [32]:
# Initialize the model (pre-trained ResNet18)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)  # Use pre-trained weights

# Modify the final fully connected layer to match the number of classes in your dataset
model.fc = nn.Linear(model.fc.in_features, len(dataset.classes))

# Move the model to the GPU (if available)
model = model.to(device)


# Unfreeze all layers (i.e., make them trainable)
for param in model.parameters():
    param.requires_grad = True


In [33]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Learning rate can be adjusted
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, verbose=True)


In [34]:
# Training loop
num_epochs = 40

for epoch in range(num_epochs):
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated(device) / 1e9} GB")
    model.train()  # Set model to training mode
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader)}")

    # Validation phase
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Validation Accuracy: {accuracy:.2f}%")

    # Step the scheduler based on validation loss (optional)
    scheduler.step(1 - accuracy)

GPU Memory Allocated: 0.171664896 GB
Epoch [1/40], Loss: 3.15182935500145
Validation Accuracy: 43.10%
GPU Memory Allocated: 0.215821824 GB
Epoch [2/40], Loss: 1.7328266628980638
Validation Accuracy: 54.95%
GPU Memory Allocated: 0.215821824 GB
Epoch [3/40], Loss: 1.194388749241829
Validation Accuracy: 59.17%
GPU Memory Allocated: 0.215821824 GB
Epoch [4/40], Loss: 0.8020116168260575
Validation Accuracy: 58.88%
GPU Memory Allocated: 0.215821824 GB
Epoch [5/40], Loss: 0.5312895797789097
Validation Accuracy: 62.23%
GPU Memory Allocated: 0.215821824 GB
Epoch [6/40], Loss: 0.3663561129495502
Validation Accuracy: 61.38%
GPU Memory Allocated: 0.215821824 GB
Epoch [7/40], Loss: 0.2691670284792781
Validation Accuracy: 61.45%
GPU Memory Allocated: 0.215821824 GB
Epoch [8/40], Loss: 0.22688875981047749
Validation Accuracy: 61.83%
GPU Memory Allocated: 0.215821824 GB
Epoch [9/40], Loss: 0.0818768624975346
Validation Accuracy: 69.65%
GPU Memory Allocated: 0.215821824 GB
Epoch [10/40], Loss: 0.023370

In [35]:
torch.save(model.state_dict(), 'resnet18_trained_modelv2.pth')